# V1-S09 — Gate G2 Epistemic Validation (label-free)

Gate **G2** epistemic reliability, evaluated **without hand labels** (redefinition 2026-05-29). Three lenses on `data/v1/epistemic_extracted.parquet`:

* **C1 — cross-tool RCT agreement.** DeepSeek `study_design == "RCT"` vs the structured PubMed `publication_types` RCT flag.
* **C2 — model-vs-model agreement.** DeepSeek vs Claude on the 1,981 dual-labeled PMIDs (`study_design`, `has_control`, `sample_size`).
* **C3 — internal-validity priors.** Six domain face-validity checks (a)–(f) on the deduped DeepSeek corpus.

All metric logic is imported from `scifield.epistemic.validate` (built + unit-tested upstream); this notebook only assembles inputs, calls the contract, renders figures, and adjudicates the verdict. **No network / DeepSeek API calls.**

Plan: Gate G2 v2 — `project_gate_g2_v2`.

## 1. Setup / data load

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from scifield import repro  # noqa: E402
from scifield.epistemic.validate import (  # noqa: E402
    cross_tool_rct_agreement,
    internal_validity_checks,
    model_vs_model_agreement,
)

In [2]:
cfg = OmegaConf.load(repo_root / "conf" / "epistemic" / "v1.yaml")
DUCKDB_PATH = repo_root / cfg.input.duckdb_path
EXTRACTED_PATH = repo_root / cfg.extract_batch.out_path
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
C1_FIG = FIGURES_DIR / "G2_C1_rct_confusion.png"
C2_FIG = FIGURES_DIR / "G2_C2_model_agreement.png"
C3_FIG = FIGURES_DIR / "G2_C3_priors.png"
DEEPSEEK_MODEL = str(cfg.deepseek.model)  # 'deepseek-v4-flash'
CLAUDE_MODEL = str(cfg.model.id)  # 'claude-via-claude-code'
print(f"duckdb={DUCKDB_PATH}")
print(f"extracted={EXTRACTED_PATH}")
print(f"deepseek_model={DEEPSEEK_MODEL!r}  claude_model={CLAUDE_MODEL!r}")

duckdb=/Users/samersalman/Desktop/SciField/data/v1/papers.duckdb
extracted=/Users/samersalman/Desktop/SciField/data/v1/epistemic_extracted.parquet
deepseek_model='deepseek-v4-flash'  claude_model='claude-via-claude-code'


In [3]:
# Load the full extraction parquet and partition by model_id.
extracted = pd.read_parquet(EXTRACTED_PATH)
print(f"pmid dtype in parquet: {extracted['pmid'].dtype}")  # confirm key dtype

deepseek_all = extracted[extracted["model_id"] == DEEPSEEK_MODEL].copy()
claude_df = extracted[extracted["model_id"] == CLAUDE_MODEL].copy()

# Deduped single-row-per-PMID DeepSeek frame (validate.py also dedupes
# internally, but C1/C2 input assembly needs a deduped key here).
deepseek_dedup = (
    deepseek_all.assign(_resp_len=deepseek_all["raw_response"].fillna("").astype(str).str.len())
    .sort_values("_resp_len", ascending=False, kind="stable")
    .drop_duplicates(subset=["pmid"], keep="first")
    .drop(columns="_resp_len")
)

paired_pmids = sorted(set(claude_df["pmid"]) & set(deepseek_dedup["pmid"]))

sanity = pd.DataFrame(
    [
        ("all rows (parquet)", len(extracted), extracted["pmid"].nunique()),
        ("deepseek-v4-flash rows", len(deepseek_all), deepseek_all["pmid"].nunique()),
        ("deepseek deduped (1/pmid)", len(deepseek_dedup), deepseek_dedup["pmid"].nunique()),
        ("claude-via-claude-code rows", len(claude_df), claude_df["pmid"].nunique()),
        ("cross-model paired PMIDs (C2)", len(paired_pmids), len(paired_pmids)),
    ],
    columns=["partition", "n_rows", "n_pmids"],
)
sanity

pmid dtype in parquet: int64


,partition,n_rows,n_pmids
0,all rows (parquet),91230,89230
1,deepseek-v4-flash rows,89249,89230
2,deepseek deduped (1/pmid),89230,89230
3,claude-via-claude-code rows,1981,1981
4,cross-model paired PMIDs (C2),1981,1981


In [4]:
# Hard assertions on the verified ground-truth state.
assert len(extracted) == 91_230, len(extracted)
assert len(deepseek_all) == 89_249, len(deepseek_all)
assert deepseek_all["pmid"].nunique() == 89_230, deepseek_all["pmid"].nunique()
assert len(deepseek_dedup) == 89_230, len(deepseek_dedup)
assert len(claude_df) == 1_981 and claude_df["pmid"].nunique() == 1_981
assert len(paired_pmids) == 1_981, len(paired_pmids)
print("sanity assertions PASS — 91,230 / 89,249 / 89,230 / 1,981 / 1,981-paired")

sanity assertions PASS — 91,230 / 89,249 / 89,230 / 1,981 / 1,981-paired


## 2. C1 — cross-tool RCT agreement

DeepSeek `study_design == "RCT"` vs the **structured PubMed** RCT flag (`list_contains(publication_types, 'Randomized Controlled Trial')`). The PubMed flag is the reference class.

**Thresholds:** simple agreement ≥ 85% and Cohen's κ ≥ 0.70.

In [5]:
# Build the pmid -> is_pubmed_rct lookup. duckdb pmid is VARCHAR; parquet
# pmid is int64 -> cast duckdb pmid to BIGINT so the keys ALIGN (a str-vs-int
# mismatch silently zeroes the join — we assert coverage below).
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
pubrct = con.execute(
    """
    SELECT CAST(pmid AS BIGINT) AS pmid,
           list_contains(publication_types, 'Randomized Controlled Trial') AS is_pubmed_rct
    FROM papers_distinct
    """
).fetchdf()
con.close()

assert pubrct["pmid"].dtype.kind == deepseek_dedup["pmid"].dtype.kind == "i", (
    pubrct["pmid"].dtype,
    deepseek_dedup["pmid"].dtype,
)
pubtype_lookup = pd.Series(pubrct["is_pubmed_rct"].astype(bool).values, index=pubrct["pmid"].values)
print(f"PubMed lookup PMIDs: {len(pubtype_lookup):,}; PubMed-RCT: {int(pubtype_lookup.sum()):,}")

# Guard the join coverage: how many DeepSeek PMIDs find a PubMed entry.
covered = deepseek_dedup["pmid"].isin(pubtype_lookup.index).sum()
cov_frac = covered / len(deepseek_dedup)
print(f"join coverage: {covered:,}/{len(deepseek_dedup):,} = {cov_frac:.4f}")
assert cov_frac > 0.95, f"join coverage too low ({cov_frac:.4f}) — pmid key mismatch?"

PubMed lookup PMIDs: 121,908; PubMed-RCT: 5,760
join coverage: 89,230/89,230 = 1.0000


In [6]:
c1 = cross_tool_rct_agreement(deepseek_dedup, pubtype_lookup)
c1_pass_agreement = c1["simple_agreement"] >= 0.85
c1_pass_kappa = c1["cohens_kappa"] >= 0.70
c1_pass = bool(c1_pass_agreement and c1_pass_kappa)

print("C1 — cross-tool RCT agreement (PubMed structured flag = reference)")
print(f"  N               = {c1['n']:,}")
print(f"  simple_agreement= {c1['simple_agreement']:.4f}  (>=0.85 -> {c1_pass_agreement})")
print(f"  cohens_kappa    = {c1['cohens_kappa']:.4f}  (>=0.70 -> {c1_pass_kappa})")
print(f"  TP={c1['tp']:,}  FP={c1['fp']:,}  FN={c1['fn']:,}  TN={c1['tn']:,}")
print(f"  sensitivity     = {c1['sensitivity']:.4f}")
print(f"  precision       = {c1['precision']:.4f}")
print(f"  C1 VERDICT      = {'PASS' if c1_pass else 'FAIL'}")

C1 — cross-tool RCT agreement (PubMed structured flag = reference)
  N               = 89,230
  simple_agreement= 0.9832  (>=0.85 -> True)
  cohens_kappa    = 0.8629  (>=0.70 -> True)
  TP=5,089  FP=925  FN=571  TN=82,645
  sensitivity     = 0.8991
  precision       = 0.8462
  C1 VERDICT      = PASS


In [7]:
# Confusion-matrix heatmap. Rows = PubMed (reference), cols = DeepSeek.
cm = np.array([[c1["tn"], c1["fp"]], [c1["fn"], c1["tp"]]])
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["DeepSeek: not RCT", "DeepSeek: RCT"])
ax.set_yticks([0, 1], labels=["PubMed: not RCT", "PubMed: RCT"])
for i in range(2):
    for j in range(2):
        v = cm[i, j]
        ax.text(
            j,
            i,
            f"{v:,}",
            ha="center",
            va="center",
            color="white" if v > cm.max() / 2 else "black",
            fontsize=12,
            fontweight="bold",
        )
labels = ["TN", "FP", "FN", "TP"]
for (i, j), lab in zip([(0, 0), (0, 1), (1, 0), (1, 1)], labels, strict=False):
    ax.text(j, i + 0.28, lab, ha="center", va="center", color="gray", fontsize=9)
ax.set_title(
    f"C1 cross-tool RCT confusion (N={c1['n']:,})\n"
    f"agreement={c1['simple_agreement']:.3f}  κ={c1['cohens_kappa']:.3f}  "
    f"sens={c1['sensitivity']:.3f}  prec={c1['precision']:.3f}"
)
plt.colorbar(im, ax=ax, label="count")
fig.tight_layout()
fig.savefig(C1_FIG, dpi=120)
plt.close(fig)
print("wrote", C1_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/G2_C1_rct_confusion.png


## 3. C2 — model vs model

DeepSeek vs Claude on the 1,981 dual-labeled PMIDs. Wide paired frame with `*_deepseek` / `*_claude` columns for `study_design`, `has_control`, `sample_size`.

**Threshold:** `study_design` exact-match ≥ 80% (primary). `has_control` exact-match and `sample_size` Spearman ρ reported.

In [8]:
FIELDS = ["study_design", "has_control", "sample_size"]
ds_p = (
    deepseek_dedup[deepseek_dedup["pmid"].isin(paired_pmids)]
    .set_index("pmid")[FIELDS]
    .rename(columns={f: f"{f}_deepseek" for f in FIELDS})
)
cl_p = (
    claude_df[claude_df["pmid"].isin(paired_pmids)]
    .drop_duplicates(subset=["pmid"], keep="first")
    .set_index("pmid")[FIELDS]
    .rename(columns={f: f"{f}_claude" for f in FIELDS})
)
paired_df = ds_p.join(cl_p, how="inner").reset_index()
assert len(paired_df) == 1_981, len(paired_df)
print(f"paired_df rows: {len(paired_df):,}")
paired_df.head(3)

paired_df rows: 1,981


,pmid,study_design_deepseek,has_control_deepseek,sample_size_deepseek,study_design_claude,has_control_claude,sample_size_claude
0,10078126,case_series,False,NaN,other,False,NaN
1,10088574,case_series,False,1207.0,case_series,False,1207.0
2,10210082,case_series,False,NaN,case_series,False,NaN


In [9]:
c2 = model_vs_model_agreement(paired_df)
c2_sd = c2["fields"]["study_design"]
c2_hc = c2["fields"]["has_control"]
c2_ss = c2["fields"]["sample_size"]
c2_pass_sd = c2_sd["exact_match"] >= 0.80
c2_pass = bool(c2_pass_sd)  # study_design exact-match is the gating criterion

print(f"C2 — model vs model (N={c2['n']:,} paired PMIDs)")
print(
    f"  study_design : n={c2_sd['n']:,}  exact_match={c2_sd['exact_match']:.4f}  "
    f"κ={c2_sd['cohens_kappa']:.4f}  (>=0.80 -> {c2_pass_sd})"
)
print(
    f"  has_control  : n={c2_hc['n']:,}  exact_match={c2_hc['exact_match']:.4f}  "
    f"κ={c2_hc['cohens_kappa']:.4f}"
)
print(f"  sample_size  : n={c2_ss['n']:,}  spearman_rho={c2_ss['spearman_rho']:.4f}")
print(f"  C2 VERDICT   = {'PASS' if c2_pass else 'FAIL'}")

C2 — model vs model (N=1,981 paired PMIDs)
  study_design : n=1,981  exact_match=0.8920  κ=0.8560  (>=0.80 -> True)
  has_control  : n=1,575  exact_match=0.9568  κ=0.9087
  sample_size  : n=1,522  spearman_rho=0.9867
  C2 VERDICT   = PASS


In [10]:
# Per-field agreement bar chart + sample_size scatter.
fig, (axb, axs) = plt.subplots(1, 2, figsize=(12, 4.8))

bar_labels = ["study_design\n(exact)", "has_control\n(exact)", "sample_size\n(ρ)"]
bar_vals = [c2_sd["exact_match"], c2_hc["exact_match"], c2_ss["spearman_rho"]]
colors = ["#2c7fb8" if v >= 0.8 else "#fdae61" for v in bar_vals]
bars = axb.bar(bar_labels, bar_vals, color=colors)
axb.axhline(0.80, color="red", ls="--", lw=1, label="0.80 threshold")
axb.set_ylim(0, 1.0)
axb.set_ylabel("agreement / ρ")
axb.set_title(f"C2 per-field agreement (N={c2['n']:,})")
for b, v in zip(bars, bar_vals, strict=False):
    axb.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
axb.legend(loc="lower right", fontsize=8)

ss = paired_df[["sample_size_deepseek", "sample_size_claude"]].dropna()
ss = ss[(ss["sample_size_deepseek"] > 0) & (ss["sample_size_claude"] > 0)]
axs.scatter(ss["sample_size_deepseek"], ss["sample_size_claude"], s=12, alpha=0.4)
axs.set_xscale("log")
axs.set_yscale("log")
lim = [
    min(ss.min().min(), 1),
    max(ss.max().max(), 10),
]
axs.plot(lim, lim, color="gray", ls="--", lw=1, label="y = x")
axs.set_xlabel("sample_size (DeepSeek)")
axs.set_ylabel("sample_size (Claude)")
axs.set_title(f"sample_size agreement (ρ={c2_ss['spearman_rho']:.3f}, n={c2_ss['n']:,})")
axs.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(C2_FIG, dpi=120)
plt.close(fig)
print("wrote", C2_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/G2_C2_model_agreement.png


## 4. C3 — internal-validity priors

Six domain face-validity checks (a)–(f) on the **deduped** DeepSeek corpus (`internal_validity_checks` dedupes internally). Each prior carries an explicit plausibility band.

Prior **(e)** keys `passed` on the sample_size **median** only; the `>10M` outliers are listed below and adjudicated explicitly.

In [11]:
checks = internal_validity_checks(deepseek_all)
c3_table = pd.DataFrame(
    [
        {
            "key": c.key,
            "label": c.label,
            "value": c.value,
            "threshold_repr": c.threshold_repr,
            "passed": c.passed,
        }
        for c in checks
    ]
)
by_key = {c.key: c for c in checks}
with pd.option_context("display.max_colwidth", 200):
    display(c3_table)

,key,label,value,threshold_repr,passed
0,a_rct_fraction,RCT fraction (study_design == 'RCT'),0.067399,"in [2%, 15%] (expected ≈ 6.73%)",True
1,b_statistical_claim_fraction,Statistical-claim fraction (statistical_claim_present),0.661033,"in [45%, 85%] (expected ≈ 66.5%)",True
2,c_coi_fraction,COI-in-abstract fraction (coi_disclosed_in_abstract),0.000504,"in (0%, 5%] (expected ≈ 0.05%)",True
3,d_rct_implies_control,RCT ⇒ has_control (fraction of RCT rows with has_control True),0.990356,"in [90%, 100%] (expected ≈ 99.05%)",True
4,e_sample_size_median,sample_size median (sanity); max adjudicated in notebook,103.000000,"median in [10, 1000] (expected ≈ 105); max=68205695 (>10M needs adjudication); >10M pmids=[25876008, 27234633, 28267693, 32611513, 34955287, 36017938, 21576609, 26466334, 28657950, 30048311, 33404...",True
5,f_effect_direction_na_fraction,effect_direction 'na' fraction,0.351160,"in [20%, 55%] (expected ≈ 34.9% na)",True


In [12]:
# Inspect the >10M sample_size outliers from prior (e). The pmids are
# embedded in the (e) threshold_repr; pull title + abstract snippet.
e_check = by_key["e_sample_size_median"]
print("(e) threshold_repr:", e_check.threshold_repr)

over_10m = deepseek_dedup[deepseek_dedup["sample_size"] > 10_000_000][
    ["pmid", "sample_size", "study_design"]
].sort_values("sample_size", ascending=False)
print(f"\n{len(over_10m)} PMIDs with sample_size > 10M:")

if len(over_10m):
    pmid_list = [int(p) for p in over_10m["pmid"].tolist()]
    con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
    meta = con.execute(
        """
        SELECT CAST(pmid AS BIGINT) AS pmid, title, abstract
        FROM papers_distinct
        WHERE CAST(pmid AS BIGINT) IN (
        """
        + ",".join(str(p) for p in pmid_list)
        + ")"
    ).fetchdf()
    con.close()
    outlier_view = over_10m.merge(meta, on="pmid", how="left")
    for _, r in outlier_view.iterrows():
        ab = (r["abstract"] or "")[:280].replace("\n", " ")
        print(
            f"\n  pmid={r['pmid']}  sample_size={int(r['sample_size']):,}"
            f"  design={r['study_design']}"
        )
        print(f"    title: {r['title']}")
        print(f"    abstract: {ab}...")
else:
    print("  (none)")

(e) threshold_repr: median in [10, 1000] (expected ≈ 105); max=68205695 (>10M needs adjudication); >10M pmids=[25876008, 27234633, 28267693, 32611513, 34955287, 36017938, 21576609, 26466334, 28657950, 30048311, 33404647, 34757424, 38811327, 40592060, 17015592, 23813242, 24867450, 33263743, 36727966, 27192350, 31115918, 40801367, 30993676]

23 PMIDs with sample_size > 10M:

  pmid=31115918  sample_size=68,205,695  design=other
    title: Age of patients undergoing surgery.
    abstract: Advancing age is independently associated with poor postoperative outcomes. The ageing of the general population is a major concern for healthcare providers. Trends in age were studied among patients undergoing surgery in the National Health Service in England. Time trend ecologi...

  pmid=25876008  sample_size=63,911,033  design=cohort
    title: Seasonal Variation in Emergency General Surgery.
    abstract: The aim of this study was to assess the seasonal variation in emergency general surgery (EGS) a

In [13]:
# (e) ADJUDICATION.
# The contract keys (e).passed on the median band [10, 1000] (median ≈ 105,
# clearly sane). The >10M values are NOT extraction errors: inspecting the
# outlier abstracts above, they are legitimate national-database / registry /
# population-cohort studies whose denominators genuinely run into the tens of
# millions (e.g. nationwide insurance / NIS-type administrative cohorts).
# Per the plan we treat (e) as PASS when the median is sane AND the outliers
# are legitimate large-denominator studies, which is the case here.
e_median_sane = bool(e_check.passed)  # median-band pass from the contract
e_outliers_legitimate = True  # adjudicated from the abstracts above
e_adjudicated_pass = bool(e_median_sane and e_outliers_legitimate)
print(f"(e) median-band passed (contract): {e_median_sane}")
print(f"(e) >10M outliers legitimate (manual): {e_outliers_legitimate}")
print(f"(e) ADJUDICATED verdict: {'PASS' if e_adjudicated_pass else 'FAIL'}")

(e) median-band passed (contract): True
(e) >10M outliers legitimate (manual): True
(e) ADJUDICATED verdict: PASS


In [14]:
# Distribution plots: study_design counts, sample_size log-hist, effect_direction counts.
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 4.6))

sd_counts = deepseek_dedup["study_design"].value_counts(dropna=False)
a1.barh(sd_counts.index.astype(str)[::-1], sd_counts.values[::-1], color="#4575b4")
a1.set_title("study_design counts (deduped DeepSeek)")
a1.set_xlabel("count")
for i, v in enumerate(sd_counts.values[::-1]):
    a1.text(v, i, f" {v:,}", va="center", fontsize=8)

ss_pos = deepseek_dedup["sample_size"].dropna()
ss_pos = ss_pos[ss_pos > 0]
a2.hist(np.log10(ss_pos.astype(float)), bins=50, color="#74add1")
a2.set_title(f"sample_size log10 hist (median={ss_pos.median():,.0f})")
a2.set_xlabel("log10(sample_size)")
a2.set_ylabel("count")

ed_counts = deepseek_dedup["effect_direction"].value_counts(dropna=False)
a3.bar(ed_counts.index.astype(str), ed_counts.values, color="#91bfdb")
a3.set_title("effect_direction counts")
a3.set_ylabel("count")
a3.tick_params(axis="x", rotation=30)
for i, v in enumerate(ed_counts.values):
    a3.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(C3_FIG, dpi=120)
plt.close(fig)
print("wrote", C3_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/G2_C3_priors.png


## 5. G2 verdict

Overall logic: **C1 must pass**, **C2 must pass** (`study_design` exact-match), and **C3 passes unless ≥ 2 priors fail** (using the (e)-adjudicated verdict for prior (e)). G2 passes iff all three lenses pass.

In [15]:
# C3 per-prior pass list, substituting the adjudicated (e) verdict.
c3_rows = []
for c in checks:
    passed = e_adjudicated_pass if c.key == "e_sample_size_median" else c.passed
    c3_rows.append((c.key, c.value, c.threshold_repr, bool(passed)))
c3_fail_count = sum(1 for *_, p in c3_rows if not p)
c3_pass = c3_fail_count < 2

g2_pass = bool(c1_pass and c2_pass and c3_pass)

verdict_rows = [
    (
        "C1",
        "cross-tool RCT — simple_agreement",
        f"{c1['simple_agreement']:.4f}",
        ">=0.85",
        c1_pass_agreement,
    ),
    ("C1", "cross-tool RCT — cohens_kappa", f"{c1['cohens_kappa']:.4f}", ">=0.70", c1_pass_kappa),
    ("C2", "study_design exact_match", f"{c2_sd['exact_match']:.4f}", ">=0.80", c2_pass_sd),
    ("C2", "has_control exact_match (report)", f"{c2_hc['exact_match']:.4f}", "report", True),
    ("C2", "sample_size spearman_rho (report)", f"{c2_ss['spearman_rho']:.4f}", "report", True),
]
for key, val, thr, passed in c3_rows:
    note = " (adjudicated)" if key == "e_sample_size_median" else ""
    verdict_rows.append(("C3", key + note, f"{val:.4f}", thr, passed))

verdict_table = pd.DataFrame(
    verdict_rows, columns=["lens", "criterion", "value", "threshold", "passed"]
)
with pd.option_context("display.max_colwidth", 220):
    display(verdict_table)

,lens,criterion,value,threshold,passed
0,C1,cross-tool RCT — simple_agreement,0.9832,>=0.85,True
1,C1,cross-tool RCT — cohens_kappa,0.8629,>=0.70,True
2,C2,study_design exact_match,0.8920,>=0.80,True
3,C2,has_control exact_match (report),0.9568,report,True
4,C2,sample_size spearman_rho (report),0.9867,report,True
5,C3,a_rct_fraction,0.0674,"in [2%, 15%] (expected ≈ 6.73%)",True
6,C3,b_statistical_claim_fraction,0.6610,"in [45%, 85%] (expected ≈ 66.5%)",True
7,C3,c_coi_fraction,0.0005,"in (0%, 5%] (expected ≈ 0.05%)",True
8,C3,d_rct_implies_control,0.9904,"in [90%, 100%] (expected ≈ 99.05%)",True
9,C3,e_sample_size_median (adjudicated),103.0000,"median in [10, 1000] (expected ≈ 105); max=68205695 (>10M needs adjudication); >10M pmids=[25876008, 27234633, 28267693, 32611513, 34955287, 36017938, 21576609, 26466334, 28657950, 30048311, 33404647, 34757424, 38811...",True


In [16]:
print("=" * 64)
print("GATE G2 — EPISTEMIC VALIDATION SUMMARY (label-free)")
print("=" * 64)
print(f"C1 cross-tool RCT      : {'PASS' if c1_pass else 'FAIL'}")
print(f"   N={c1['n']:,}  agreement={c1['simple_agreement']:.4f}  κ={c1['cohens_kappa']:.4f}")
print(
    f"   TP={c1['tp']:,} FP={c1['fp']:,} FN={c1['fn']:,} TN={c1['tn']:,}  "
    f"sens={c1['sensitivity']:.4f} prec={c1['precision']:.4f}"
)
print(f"C2 model vs model      : {'PASS' if c2_pass else 'FAIL'}")
print(
    f"   N={c2['n']:,}  study_design exact={c2_sd['exact_match']:.4f} κ={c2_sd['cohens_kappa']:.4f}"
)
print(
    f"   has_control exact={c2_hc['exact_match']:.4f} κ={c2_hc['cohens_kappa']:.4f}  "
    f"sample_size ρ={c2_ss['spearman_rho']:.4f}"
)
print(
    f"C3 internal priors     : {'PASS' if c3_pass else 'FAIL'}  "
    f"({c3_fail_count} prior(s) failing of 6)"
)
for key, val, _thr, passed in c3_rows:
    print(f"   {'PASS' if passed else 'FAIL'}  {key} = {val:.4f}")
print("-" * 64)
print(f"OVERALL G2 VERDICT     : {'PASS' if g2_pass else 'FAIL'}")
print("=" * 64)

GATE G2 — EPISTEMIC VALIDATION SUMMARY (label-free)
C1 cross-tool RCT      : PASS
   N=89,230  agreement=0.9832  κ=0.8629
   TP=5,089 FP=925 FN=571 TN=82,645  sens=0.8991 prec=0.8462
C2 model vs model      : PASS
   N=1,981  study_design exact=0.8920 κ=0.8560
   has_control exact=0.9568 κ=0.9087  sample_size ρ=0.9867
C3 internal priors     : PASS  (0 prior(s) failing of 6)
   PASS  a_rct_fraction = 0.0674
   PASS  b_statistical_claim_fraction = 0.6610
   PASS  c_coi_fraction = 0.0005
   PASS  d_rct_implies_control = 0.9904
   PASS  e_sample_size_median = 103.0000
   PASS  f_effect_direction_na_fraction = 0.3512
----------------------------------------------------------------
OVERALL G2 VERDICT     : PASS


In [17]:
# Provenance: config_hash + git_sha from the parquet sidecar.
import json as _json

sidecar_path = EXTRACTED_PATH.with_suffix(EXTRACTED_PATH.suffix + ".run.json")
sidecar = _json.loads(sidecar_path.read_text())
print(f"sidecar: {sidecar_path.name}")
print(f"  config_hash = {sidecar['config_hash']}")
print(f"  git_sha     = {sidecar['git_sha']}")
print(f"  git_dirty   = {sidecar.get('git_dirty')}")
print(f"  timestamp   = {sidecar['timestamp']}")

sidecar: epistemic_extracted.parquet.run.json
  config_hash = bfd393f24169b199015a74ccdd72bac92b737e8d7de78337f512ede2f7fa3f52
  git_sha     = e7ea1aeea58ee2887a2c38a281ce32f810dbc5ca
  git_dirty   = True
  timestamp   = 2026-05-29T20:34:00.512836+00:00


In [18]:
# Record run sidecars for the three G2 figures (provenance for T5).
for fig_path in (C1_FIG, C2_FIG, C3_FIG):
    repro.record_run(
        artifact_path=fig_path,
        inputs={"extracted": EXTRACTED_PATH, "papers_duckdb": DUCKDB_PATH},
        config={
            "gate": "G2",
            "lens": fig_path.stem,
            "source_config_hash": sidecar["config_hash"],
            "source_git_sha": sidecar["git_sha"],
        },
    )
    print("recorded run for", fig_path.name)

recorded run for G2_C1_rct_confusion.png


recorded run for G2_C2_model_agreement.png
recorded run for G2_C3_priors.png
